# BenoStreamDB: Full Pipeline Presentation

**Version:** 0.1.12 (Alpha)  
**Project:** BenoStreamDB - Serverless Index-Streaming Data Lake

This masterclass takes you from zero to a production-grade RAG pipeline, demonstrating hardware acceleration, hybrid search, and serverless table management.

## 1. Setup & Environment

We initialize our environment and ensure all vector search dependencies are loaded.

In [1]:
import benostreamdb as bsdb
import pyarrow as pa
import pandas as pd
import numpy as np
import os, shutil, requests, time, warnings
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 150)

print("Environment ready.")

/home/ralbright/projects/benolabs/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment ready.


## 2. Hardware Acceleration

BenoStreamDB can utilize Intel GPUs for massively parallel index construction. We check for available hardware here.

GPU support for Nvidia, Intel, Apple MPS, AMD(ROCm - untested)

In [2]:
import torch
device = "cpu"
if torch.cuda.is_available():
    device = "cuda:0"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")
ctx = bsdb.Device(device)


Using device: cuda:0


## 3. Data Preparation (AG News)

We load the AG News dataset and generate semantic embeddings using a 384-dimensional SBERT model.

In [3]:
print("Loading dataset...")
try:
    dataset = load_dataset("ag_news", split="test[:500]")
except Exception:
    dataset = load_dataset("SetFit/ag_news", split="test[:500]")
df = pd.DataFrame(dataset)
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
df["category"] = df["label"].map(label_map)

print("Generating embeddings...")
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["text"].tolist())
df["embedding"] = [list(e) for e in embeddings]
print(f"Prepared {len(df)} records with {len(df['embedding'][0])}D vectors.")

df["id"] = range(len(df))


Loading dataset...


Generating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10352.58it/s]

Prepared 500 records with 384D vectors.


## 4. Table Creation (Serverless Iceberg)

BenoStreamDB tables are serverless and use standard Iceberg/Parquet formats with sidecar indices.

In [4]:
TABLE_URI = "file:///tmp/hdb_masterclass"
if os.path.exists("/tmp/hdb_masterclass"): shutil.rmtree("/tmp/hdb_masterclass")

schema = pa.schema([
    ('id', pa.int32()), 
    ('category', pa.string()),
    ('text', pa.string()),
    ('embedding', pa.list_(pa.float32(), 384))
])

table = bsdb.Table.create(TABLE_URI, schema, device=ctx)
print(f"Table created at {TABLE_URI}")

Table created at file:///tmp/hdb_masterclass


## 5. Ingestion & Indexing

We write the data and building HNSW vector indices and Inverted scalar indices simultaneously.

In [5]:
table.add_index_columns(["embedding", "category", "text"])
table.write(df[["id", "category", "text", "embedding"]])
table.commit()
print("Data ingested and indexed successfully.")

Data ingested and indexed successfully.


## 6. Hybrid Search Performance

We perform a semantic search combined with a scalar filter to find 'Space' news in the 'Sci/Tech' category.

In [6]:
query_vec = list(model.encode("Latest milestones in space exploration"))

results = (table.filter("category = 'Sci/Tech'")
                 .vector_search(query_vec, k=3)
                 .to_pandas())

print("Hybrid Search Results (Sci/Tech + Vector):")
display(results[["category", "text", "distance"]])

Hybrid Search Results (Sci/Tech + Vector):


,category,text,distance
0,Sci/Tech,"The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeers ...",1.113737
1,Sci/Tech,Stunt pilots to snag a falling NASA craft NASA #39;s three-year effort to bring some genuine star dust back to Earth is set for a dramatic finale ...,1.254462
2,Sci/Tech,Mars Rovers Relay Images Through Mars Express European Space Agency -- ESAs Mars Express has relayed pictures from one of NASA's Mars rovers for t...,1.255060


## 7. SQL & pgvector Interface

BenoStreamDB supports standard SQL with pgvector-compatible operators for vector distance.

In [7]:
session = bsdb.Session()
session.register("news", table)

query_vec_str = str([float(x) for x in query_vec]).replace(' ', '')
sql_df = session.sql(f"""
    SELECT category, text, 
           embedding <-> '{query_vec_str}'::vector AS dist
    FROM news
    WHERE category = 'Sci/Tech'
    ORDER BY dist ASC LIMIT 3
""")
display(sql_df.to_pandas())

,category,text,dist
0,Sci/Tech,"The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeers ...",1.055337
1,Sci/Tech,Stunt pilots to snag a falling NASA craft NASA #39;s three-year effort to bring some genuine star dust back to Earth is set for a dramatic finale ...,1.120027
2,Sci/Tech,Mars Rovers Relay Images Through Mars Express European Space Agency -- ESAs Mars Express has relayed pictures from one of NASA's Mars rovers for t...,1.120294


## 8. Vector Aggregations

Calculate semantic centroids for entire categories directly in SQL.

In [8]:
agg_df = session.sql("""
    SELECT category, 
           vector_avg(embedding) AS centroid,
           COUNT(*) as count
    FROM news
    GROUP BY category
    order by count asc
""").to_pandas()
display(agg_df[["category", "count"]])

,category,count
0,Business,106
1,World,122
2,Sci/Tech,127
3,Sports,145


## 9. Table Maintenance

Keep the data lake healthy by compacting small files and vacuuming history.

In [9]:
# Access the file count directly from the table statistics property
print(f"Files before: {table.statistics.file_count}")
table.compact()
table.commit()
print(f"Files after: {table.statistics.file_count}")

table.vacuum(retention_versions=1)
print("Vacuum complete.")

Files before: 1


Files after: 1
Vacuum complete.


## 10. Advanced Partitioning

Scale to massive datasets using identity partitioning on the category field.

In [10]:
PART_URI = "file:///tmp/part_demo"
if os.path.exists("/tmp/part_demo"): shutil.rmtree("/tmp/part_demo")

partition_spec = {
    "fields": [{"name": "category", "transform": "identity", "source_id": 1, "field_id": 1000}]
}

part_table = bsdb.Table.create_partitioned(PART_URI, schema, partition_spec)
part_table.write(df[:100])
part_table.commit()

print("Partitioned physical layout:")
# Use dictionary indexing for both the manifest and the individual entries
for entry in part_table.manifest()["entries"]:
    print(f" - {entry['file_path'].split('/')[-2:]}")

Partitioned physical layout:


## 11. End-to-End RAG Pipeline

Using SQuAD validation data to build a knowledge-backed Q&A system.

In [11]:
print("Loading SQuAD contexts...")
try:
    squad = load_dataset("squad", split="validation[0:100]")
except Exception:
    squad = load_dataset("rajpurkar/squad", split="validation[0:100]")
squad_df = pd.DataFrame(squad).drop_duplicates(subset=["context"])

squad_emb = model.encode(squad_df["context"].tolist(), show_progress_bar=True)
squad_df["embedding"] = [list(e) for e in squad_emb]

rag_table = bsdb.Table("rag_db_presentation")
rag_table.add_index_columns(["embedding", "context"])
rag_table.write(squad_df[["context", "title", "embedding"]])
rag_table.commit()
print("RAG knowledge base initialized.")

Loading SQuAD contexts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 153.58it/s]

RAG knowledge base initialized.


## 12. Inference with Retrieval

Connect retrieval to the LLM (requires GROQ_API_KEY).

In [12]:
def ask_hdb(question):
    # Convert question to an embedding (using your local model)
    q_emb = model.encode(question)
    
    # Perform a vector search on your table
    ctx_df = rag_table.to_pandas(vector_filter={
        "column": "embedding", 
        "query": q_emb, 
        "k": 3  # Get top 3 matching contexts
    })
    
    # Combine retrieved contexts into one string
    contexts = ctx_df["context"].tolist()
    prompt = f"Answer the question based ONLY on this context:\n\n" + "\n".join(contexts) + f"\n\nQuestion: {question}"
    
    # Configure OpenCode Go endpoint
    base_url = os.environ.get("OPENCODE_BASE_URL", "https://opencode.ai/zen/go/v1")
    api_key = os.environ.get("OPENCODE_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
    model_name = os.environ.get("OPENCODE_MODEL", "deepseek-v4.1-flash")
    headers = {"Authorization": f"Bearer {api_key}"}
    if "opencode.ai" in base_url:
        headers["x-opencode-session"] = "benolabs-howto"
    
    try:
        if not api_key:
            raise ValueError("No active API key provided")
        r = requests.post(
            f"{base_url.rstrip('/')}/chat/completions", 
            headers=headers,
            json={
                "model": model_name, 
                "messages": [{"role": "user", "content": prompt}],
                "thinking": {"type": "disabled"}
            },
            timeout=30
        )
        res = r.json()
        if "choices" in res:
            return res["choices"][0]["message"]["content"].strip()
        print(f"API Error: {res}")
        return contexts[0]
    except Exception as e:
        print(f"[Notice] LLM fallback: {e}")
        return contexts[0]

In [13]:

question = "Where were the first modern Olympic games?"
print(f"Q: {question}\nA: {ask_hdb(question)}")

Q: Where were the first modern Olympic games?
A: The context does not provide information about where the first modern Olympic games were held.


In [14]:
question = "What city was superbowl 50 played at?"
print(f"Q: {question}\nA: {ask_hdb(question)}")

Q: What city was superbowl 50 played at?
A: Santa Clara, California
